In [2]:
import sys

sys.path.append("..")  # lets us import from src/

from collections import Counter
import pandas as pd
from PIL import Image
from torch.utils.data import DataLoader

from src.dataset import ECGDataset, get_eval_transform, CLASS_NAMES

# 1) Check the raw image sizes (to decide resize strategy later)
folds_df = pd.read_csv("../data/folds.csv")
sizes = Counter()
for p in folds_df["path"]:
    with Image.open("../" + p) as img:
        sizes[img.size] += 1
print("Raw image sizes:", sizes.most_common(5))

# 2) Build train (folds 0-3) and validation (fold 4) datasets
train_ds = ECGDataset(
    "../data/folds.csv", "..", folds=[0, 1, 2, 3], transform=get_eval_transform()
)
val_ds = ECGDataset(
    "../data/folds.csv", "..", folds=[4], transform=get_eval_transform()
)
print("train:", len(train_ds), "| val:", len(val_ds))

# 3) One sample
image, label = train_ds[0]
print("image shape:", image.shape, "| label:", label, CLASS_NAMES[label])

# 4) One batch from a DataLoader
loader = DataLoader(train_ds, batch_size=16, shuffle=True)
images, labels = next(iter(loader))
print("batch shape:", images.shape)
print("batch labels:", labels)

Raw image sizes: [((2213, 1572), 491)]
train: 393 | val: 98
image shape: torch.Size([3, 224, 224]) | label: 2 MI
batch shape: torch.Size([16, 3, 224, 224])
batch labels: tensor([0, 1, 0, 0, 0, 0, 1, 3, 0, 1, 0, 0, 1, 3, 0, 3])
